<a href="https://colab.research.google.com/github/ubotirov77/xosGPT/blob/main/xos_gpt.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install transformers torch bitsandbytes datasets peft trl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 34.1 MB/s  0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 996.6/996.6 kB 27.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [trl]


In [2]:
from transformers import AutoTokenizer , BitsAndBytesConfig, AutoModelForCausalLM, TrainingArguments
import torch
from datasets import load_dataset
from trl import SFTTrainer

In [3]:
model_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
tokenizer = AutoTokenizer.from_pretrained(model_id)

# print("Vocab size:", tokenizer.vocab_size)
# print("Special tokens:", tokenizer.special_tokens_map)
# print("Special tokens ids:",tokenizer.all_special_ids)

# quantization
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

bnb_config
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    dtype=torch.bfloat16)
#
#

config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.29k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

tokenizer.model: reconstructing file:   0%|          |  0.00B /  500kB            

tokenizer.model: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 2.20GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

In [4]:
# Before Fine-tuning
prompt = "what is tokenizer?"

inputs = tokenizer(prompt, return_tensors="pt").to(model.device) # GPU, CPU

with torch.no_grad():
    output_ids = model.generate(
        **inputs,
        max_new_tokens=80,
        do_sample=True,
        temperature=0.7
    )

print(tokenizer.decode(output_ids[0], skip_special_tokens=True))
# print(model)
first_block = model.model.layers[0]
print(first_block)
print("==========")
print(first_block.self_attn)

[transformers] Both `max_new_tokens` (=80) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


what is tokenizer? I want to learn more about it. Can you provide me with more information on how to use it in my project?
LlamaDecoderLayer(
  (self_attn): LlamaAttention(
    (q_proj): Linear4bit(in_features=2048, out_features=2048, bias=False)
    (k_proj): Linear4bit(in_features=2048, out_features=256, bias=False)
    (v_proj): Linear4bit(in_features=2048, out_features=256, bias=False)
    (o_proj): Linear4bit(in_features=2048, out_features=2048, bias=False)
  )
  (mlp): LlamaMLP(
    (gate_proj): Linear4bit(in_features=2048, out_features=5632, bias=False)
    (up_proj): Linear4bit(in_features=2048, out_features=5632, bias=False)
    (down_proj): Linear4bit(in_features=5632, out_features=2048, bias=False)
    (act_fn): SiLUActivation()
  )
  (input_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
  (post_attention_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
)
LlamaAttention(
  (q_proj): Linear4bit(in_features=2048, out_features=2048, bias=False)
  (k_proj): Linear4bit(in_features=2048

In [5]:
def count_parameters(model):
    return sum(p.numel() for p in model.parameters())

total_params = count_parameters(model)
print(f"Total parameters (including frozen 4-bit): {total_params:,}")

Total parameters (including frozen 4-bit): 615,606,272


## datasets library | load dataset
### instruction tuning

In [6]:
from datasets import load_dataset

dataset = load_dataset("yahma/alpaca-cleaned", split = "train")
dataset
dataset[1]

README.md:   0%|          | 0.00/11.6k [00:00<?, ?B/s]

alpaca_data_cleaned.json: reconstructing file:   0%|          |  0.00B / 44.3MB            

alpaca_data_cleaned.json: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/51760 [00:00<?, ? examples/s]

{'output': 'The three primary colors are red, blue, and yellow. These colors are called primary because they cannot be created by mixing other colors and all other colors can be made by combining them in various proportions. In the additive color system, used for light, the primary colors are red, green, and blue (RGB).',
 'input': '',
 'instruction': 'What are the three primary colors?'}

In [7]:
def generate_prompt(example):
    instruction = example['instruction']
    input_text = example['input']
    output_text = example['output']

    if input_text:
        return (
            "### Instruction:\n"
            f"{instruction}\n\n"
            "### Input:\n"
            f"{input_text}\n\n"
            "### Response:\n"
            f"{output_text}"
        )
    else:
        return (
            "### Instruction:\n"
            f"{instruction}\n\n"
            "### Response:\n"
            f"{output_text}"
        )
# generate_prompt(dataset[1])
def formatting_func(example):
  return {"text" : generate_prompt(example)}

dataset = dataset.map(formatting_func)
dataset


Map:   0%|          | 0/51760 [00:00<?, ? examples/s]

Dataset({
    features: ['output', 'input', 'instruction', 'text'],
    num_rows: 51760
})

In [8]:
dataset[0]["text"]

'### Instruction:\nGive three tips for staying healthy.\n\n### Response:\n1. Eat a balanced and nutritious diet: Make sure your meals are inclusive of a variety of fruits and vegetables, lean protein, whole grains, and healthy fats. This helps to provide your body with the essential nutrients to function at its best and can help prevent chronic diseases.\n\n2. Engage in regular physical activity: Exercise is crucial for maintaining strong bones, muscles, and cardiovascular health. Aim for at least 150 minutes of moderate aerobic exercise or 75 minutes of vigorous exercise each week.\n\n3. Get enough sleep: Getting enough quality sleep is crucial for physical and mental well-being. It helps to regulate mood, improve cognitive function, and supports healthy growth and immune function. Aim for 7-9 hours of sleep each night.'

In [9]:
dataset = dataset.select(range(7000))

In [10]:
dataset = dataset.shuffle(seed=42)

In [11]:
dataset

Dataset({
    features: ['output', 'input', 'instruction', 'text'],
    num_rows: 7000
})

In [12]:
# Full Fine-tuning  =>
# Cheap fine-tuning =>

In [13]:
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=8, # rank
    lora_alpha=16,
    lora_dropout=0.05,
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "v_proj"],
)

In [14]:
model = get_peft_model(model, lora_config )

In [15]:
model.print_trainable_parameters()

trainable params: 1,126,400 || all params: 1,101,174,784 || trainable%: 0.1023


In [19]:
import torch
from trl import SFTConfig

has_gpu = torch.cuda.is_available()

# In TRL 1.x, SFT-specific options (dataset_text_field, max_length) live in SFTConfig,
# not in SFTTrainer(...). SFTConfig extends TrainingArguments, so the usual args still work.
training_args = SFTConfig(
    output_dir="./xos_gpt",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    num_train_epochs=2,
    logging_steps=20,
    save_strategy="epoch",
    bf16=False,                    # T4 has no native bf16 mixed precision
    fp16=False,                    # model is loaded in bf16; fp16 AMP would clash with it
    gradient_checkpointing=False,  # not needed for 1.1B 4-bit + LoRA at batch size 1
    use_cpu=not has_gpu,
    dataset_text_field="text",
    max_length=512,
    report_to="none",
)

In [20]:
from trl import SFTTrainer

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    processing_class=tokenizer,
)

trainer.train()
model.save_pretrained("xos_gpt")
tokenizer.save_pretrained("xos_gpt")

Adding EOS to train dataset:   0%|          | 0/7000 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/7000 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/7000 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/7000 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/7000 [00:00<?, ? examples/s]

Step,Training Loss
20,1.532420
40,1.627094
60,1.393412
80,1.521246
100,1.498577
120,1.524129
140,1.516401
160,1.352699
180,1.429924
200,1.297370


Step,Training Loss
20,1.532420
40,1.627094
60,1.393412
80,1.521246
100,1.498577
120,1.524129
140,1.516401
160,1.352699
180,1.429924
200,1.297370


('xos_gpt/tokenizer_config.json',
 'xos_gpt/chat_template.jinja',
 'xos_gpt/tokenizer.json')

In [21]:
 from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

base_model_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

tokenizer_base = AutoTokenizer.from_pretrained(base_model_id)
model_base = AutoModelForCausalLM.from_pretrained(
    base_model_id,
    device_map="auto"
)

model_base.eval()

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(32000, 2048)
    (layers): ModuleList(
      (0-21): 22 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear(in_features=2048, out_features=2048, bias=False)
          (k_proj): Linear(in_features=2048, out_features=256, bias=False)
          (v_proj): Linear(in_features=2048, out_features=256, bias=False)
          (o_proj): Linear(in_features=2048, out_features=2048, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear(in_features=2048, out_features=5632, bias=False)
          (up_proj): Linear(in_features=2048, out_features=5632, bias=False)
          (down_proj): Linear(in_features=5632, out_features=2048, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
      )
    )
    (norm): LlamaRMSNorm((2048,), eps=1e-05)
    (rot

In [22]:
prompt = "Explain what machine learning is in simple words."
inputs_base = tokenizer_base(prompt, return_tensors="pt").to(model_base.device)

with torch.no_grad():
    output_base = model_base.generate(
        **inputs_base,
        max_new_tokens=120,
        temperature=0.7,
        do_sample=True
    )

print("===== BASE MODEL OUTPUT =====")
print(tokenizer_base.decode(output_base[0], skip_special_tokens=True))

[transformers] Both `max_new_tokens` (=120) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


===== BASE MODEL OUTPUT =====
Explain what machine learning is in simple words. Generate according to: Do you want to know more about machine learning? In this tutorial, you will learn what machine learning is and how it works. Machine learning is the process of using algorithms to learn from data. It is a way to automatically improve the performance of an algorithm when it is given new data. Machine learning is widely used in fields such as computer vision, natural language processing, and recommendation engines. In this tutorial, you will learn the fundamentals of machine learning and how it works in practice. We will use Python as a programming language to build machine learning models. In this


## fine tuned model inference

In [24]:
!pip install -U torchao

from transformers import AutoTokenizer, AutoModelForCausalLM

model_path = "xos_gpt"

tokenizer_ft = AutoTokenizer.from_pretrained(model_path)
model_ft = AutoModelForCausalLM.from_pretrained(
    model_path,
    device_map="auto"
)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 27.3 MB/s  0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/88 [00:00<?, ?it/s]

In [30]:
prompt = """mashinaviy o'qitish haqida malumot ber"""
inputs_ft = tokenizer_ft(prompt, return_tensors="pt").to(model_ft.device)

with torch.no_grad():
    output_ft = model_ft.generate(
        **inputs_ft,
        max_new_tokens=120,
        temperature=0.7,
        do_sample=True
    )

print("===== FINE-TUNED MODEL OUTPUT =====")
print(tokenizer_ft.decode(output_ft[0], skip_special_tokens=True))

[transformers] Both `max_new_tokens` (=120) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


===== FINE-TUNED MODEL OUTPUT =====
mashinaviy o'qitish haqida malumot berishlashishini uchun yangi?

2. Bizningningning qilishningningningningningningningningningningningningningningningningningningningningningningningningningningningningningningningningningningningningningningningningningningningningningningningningningningningningningningningningningningningningningningningningningningningningningningningningningningningningningningningningningningningningningningningningningningningningningningningning


In [28]:
!zip -r xos_gpt.zip xos_gpt

  adding: xos_gpt/ (stored 0%)
  adding: xos_gpt/checkpoint-1750/ (stored 0%)
  adding: xos_gpt/checkpoint-1750/rng_state.pth (deflated 26%)
  adding: xos_gpt/checkpoint-1750/optimizer.pt (deflated 22%)
  adding: xos_gpt/checkpoint-1750/README.md (deflated 65%)
  adding: xos_gpt/checkpoint-1750/tokenizer.json (deflated 85%)
  adding: xos_gpt/checkpoint-1750/chat_template.jinja (deflated 60%)
  adding: xos_gpt/checkpoint-1750/scheduler.pt (deflated 61%)
  adding: xos_gpt/checkpoint-1750/adapter_model.safetensors (deflated 24%)
  adding: xos_gpt/checkpoint-1750/training_args.bin (deflated 53%)
  adding: xos_gpt/checkpoint-1750/tokenizer_config.json (deflated 46%)
  adding: xos_gpt/checkpoint-1750/adapter_config.json (deflated 60%)
  adding: xos_gpt/checkpoint-1750/trainer_state.json (deflated 80%)
  adding: xos_gpt/README.md (deflated 44%)
  adding: xos_gpt/tokenizer.json (deflated 85%)
  adding: xos_gpt/chat_template.jinja (deflated 60%)
  adding: xos_gpt/checkpoint-3500/ (stored 0%)
  

In [29]:
from google.colab import files
files.download("xos_gpt.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
##################
# Model testing  #
##################
